In [5]:
import os, time, warnings
warnings.filterwarnings("ignore")


import logfire 
from dotenv import load_dotenv

load_dotenv()


# Verify keys
print("LOGFIRE_TOKEN  :", "✅" if os.getenv("LOGFIRE_TOKEN")  else "❌  missing")
print("GROQ_API_KEY   :", "✅" if os.getenv("GROQ_API_KEY")   else "❌  missing")
print("GEMINI_API_KEY :", "✅" if os.getenv("GEMINI_API_KEY") else "❌  missing")

LOGFIRE_TOKEN  : ✅
GROQ_API_KEY   : ✅
GEMINI_API_KEY : ✅


---
## 🧱 Part 1 — Why Logfire & First Traces

The problem with `print()` in production:

| `print()` | `logfire` |
|-----------|-----------|
| Plain string, unsearchable | Structured key-value fields, fully searchable |
| No timestamp or duration | Automatic timestamps, span duration |
| Lost in terminal noise | Real-time dashboard with filters and queries |
| Nothing in production | Persisted traces, alerting, cost analytics |

Logfire is built on **OpenTelemetry** — the industry standard. Every trace you write here is portable.

In [6]:
import logfire

logfire.configure()
logfire.info('Hello, {place}!', place='UDEMY')


20:27:43.493 Hello, UDEMY!


Logfire project URL: https://logfire-us.pydantic.dev/jangkilt/starter-project


In [7]:
logfire.configure(
    token=os.getenv("LOGFIRE_TOKEN"),
    service_name="llm-observability-course"
)

Logfire project URL: https://logfire-us.pydantic.dev/jangkilt/starter-project


In [8]:
logfire.info('Hello, {place}!', place='UDEMY')

20:29:45.710 Hello, UDEMY!


In [9]:
logfire.info("notebook_started",
            part="PART 1 - BASICS",
            instructer = "Divesh",
            tool = "Pydantic Logfire"
            )

20:30:35.448 notebook_started


### Trace

In [10]:
with logfire.span("data_processing_simulation", dataset="llm_course", rows=1000):
    logfire.info("step_started", step=1, action="loading data")
    time.sleep(0.3)

    logfire.info("step_started", step=2, action="transforming", columns=12)
    time.sleep(0.2)

    logfire.info("step_started", step=3, action="saving results", output="/tmp/out.csv")

20:34:48.446 data_processing_simulation
20:34:48.460   step_started
20:34:48.768   step_started
20:34:48.970   step_started


### 🧪 Experiment 2 — Structured Logging with Pydantic Models

The *"Pydantic"* in Pydantic Logfire: when you log a Pydantic model, Logfire **expands every field** into a searchable attribute automatically.

In a real LLM app you log request/response objects hundreds of times per minute. With string logging you get `"{'user_id': 'alice', ...}"` — unsearchable. With Logfire you get filterable columns.

In [11]:
from typing import Optional

from pydantic import BaseModel

# MOCK DATA nOT REAL DATA

class LLMRequest(BaseModel):
    user_id: str
    session_id: str
    query: str
    model: str
    temperature: float = 0.7
    max_tokens: Optional[int] = None


class LLMResponse(BaseModel):
    answer: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    model_used: str    



In [12]:
# ── Simulate logging a real LLM request/response ──────────────────────────
request = LLMRequest(
    user_id="priya",
    session_id="sess_abc123",
    query="What is Retrieval-Augmented Generation?",
    model="llama-3.3-70b-versatile",
    max_tokens=500
)

with logfire.span("llm_CALL",
                  user_id = request.user_id,
                  session_id = request.session_id,
                  model_used = request.model):
    logfire.info("request_received" , **request.model_dump())
    
    time.sleep(0.1)

    response = LLMResponse(
        answer="RAG is a technique that retrieves relevant documents...",
        input_tokens=18,
        output_tokens=120,
        latency_ms=342.5,
        model_used="llama-3.3-70b-versatile"
    )
    logfire.info("response_sent", **response.model_dump())


print(response)


21:06:06.981 llm_CALL
21:06:06.991   request_received
21:06:07.101   response_sent
answer='RAG is a technique that retrieves relevant documents...' input_tokens=18 output_tokens=120 latency_ms=342.5 model_used='llama-3.3-70b-versatile'


---
## ⚡ Part 2 — Auto-Instrumentation of LLM Calls

In Part 1 you wrote spans manually. Now let Logfire do it **automatically**.

`logfire.instrument_openai()` patches the OpenAI Python SDK.
Since **Groq** and **Gemini** both expose an OpenAI-compatible REST API, the same single line instruments all of them.

Every LLM call then automatically records:
- Model name (`llama-3.3-70b-versatile`, `gemini-2.5-flash-lite`, …)
- Input + output token counts
- Wall-clock latency
- Full prompt text and full response text

You write **zero** extra logging code. It just appears in the dashboard.

### 🧪 Experiment 3 — Instrument Groq (llama-3.3-70b)

We use `ChatOpenAI` from LangChain pointed at Groq's OpenAI-compatible endpoint.
`logfire.instrument_openai()` patches the underlying SDK — Groq calls appear in traces automatically.

```
Your code  →  ChatOpenAI(base_url="https://api.groq.com/openai/v1")
                          ↓
              [logfire.instrument_openai() intercepts here]
                          ↓
              Groq API  →  response
```

In [13]:
from langchain_openai import ChatOpenAI

from langchain_core.messages import HumanMessage


logfire.instrument_openai()

llm_groq = ChatOpenAI(
    base_url= "https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    model = "llama-3.3-70b-versatile",
    temperature=0.3
)

# Make a call — watch the trace appear in the dashboard automatically
print("Calling Groq (llama-3.3-70b)…")
response = llm_groq.invoke([
    HumanMessage(content="Explain what an observability 'span' is, in exactly 2 sentences.")
])

print(response.content)

Calling Groq (llama-3.3-70b)…
21:13:46.952 Chat Completion with 'llama-3.3-70b-versatile' [LLM]
In the context of observability, a span refers to a single, executable unit of work, such as an HTTP request or a database query, that is being measured and monitored to understand its performance and behavior. A span typically has a clear start and end time, and may be composed of multiple sub-spans, allowing for a hierarchical representation of complex workflows and enabling detailed analysis of system performance and latency.


In [14]:
# Gemini's OpenAI-compatible endpoint (no extra setup needed)
llm_gemini = ChatOpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GEMINI_API_KEY"),
    model="gemini-2.5-flash-lite",
    temperature=0.3
)


print("Calling Gemini (gemini-2.5-flash-lite)…")
try:
    response = llm_gemini.invoke([
        HumanMessage(content="Explain what an observability 'trace' is, in exactly 2 sentences.")
    ])
    print(f"\n🔵 Gemini Response:\n{response.content}")
except Exception as e:
    print(f"⚠️  Gemini call failed: {e}")
    print("    Check your GEMINI_API_KEY in .env")

Calling Gemini (gemini-2.5-flash-lite)…
21:26:32.584 Chat Completion with 'gemini-2.5-flash-lite' [LLM]

🔵 Gemini Response:
An observability trace is a record of the end-to-end journey of a request as it propagates through a distributed system. It captures the sequence of operations, their timings, and the relationships between different services involved in fulfilling that request.


In [15]:
query = "What is the difference between RAG and fine-tuning? Give 3 bullet points."

with logfire.span("model_comparison", query=query, num_models=2):

    # ── Groq ─────────────────────────────────────────────────────────────
    with logfire.span("groq_call", model="llama-3.3-70b-versatile", provider="groq"):
        t0 = time.time()
        r_groq = llm_groq.invoke([HumanMessage(content=query)])
        groq_ms = round((time.time() - t0) * 1000, 1)
        logfire.info("groq_done", latency_ms=groq_ms, answer_len=len(r_groq.content))

    # ── Gemini ────────────────────────────────────────────────────────────
    with logfire.span("gemini_call", model="gemini-2.5-flash-lite", provider="google"):
        t0 = time.time()
        try:
            r_gemini = llm_gemini.invoke([HumanMessage(content=query)])
            gemini_ms = round((time.time() - t0) * 1000, 1)
            logfire.info("gemini_done", latency_ms=gemini_ms, answer_len=len(r_gemini.content))
            gemini_answer = r_gemini.content
        except Exception as e:
            logfire.warning("gemini_failed", error=str(e))
            gemini_ms = 0
            gemini_answer = f"[Error: {e}]"

# ── Print results ─────────────────────────────────────────────────────────
print(f"🟢 Groq ({groq_ms}ms):\n{r_groq.content}")
print(f"\n🔵 Gemini ({gemini_ms}ms):\n{gemini_answer}")

21:27:40.140 model_comparison
21:27:40.142   groq_call
21:27:40.149     Chat Completion with 'llama-3.3-70b-versatile' [LLM]
21:27:41.162     groq_done
21:27:41.167   gemini_call
21:27:41.169     Chat Completion with 'gemini-2.5-flash-lite' [LLM]
21:27:46.294     gemini_done
🟢 Groq (1018.1ms):
RAG (Retrieval-Augmented Generation) and fine-tuning are two different approaches used in natural language processing (NLP) and machine learning. Here are three key differences:

* **Training objective**: Fine-tuning involves adjusting the weights of a pre-trained model to fit a specific task, whereas RAG combines a pre-trained model with a retrieval mechanism to generate text based on relevant information retrieved from a knowledge source.
* **Use of external knowledge**: RAG relies on external knowledge sources, such as databases or documents, to inform its generation, whereas fine-tuning typically relies on the patterns and relationships learned from the training data itself.
* **Flexibility a

## ChatOpenAI

In [16]:
from langchain_openai import ChatOpenAI

from langchain_core.messages import HumanMessage


logfire.instrument_openai()

llm_openai = ChatOpenAI(
    model = "gpt-4.1",
    temperature=0.3
)

# Make a call — watch the trace appear in the dashboard automatically

response = llm_openai.invoke([
    HumanMessage(content="Explain what an observability 'span' is, in exactly 2 sentences.")
])

print(response.content)

21:28:59.533 Chat Completion with 'gpt-4.1' [LLM]
An observability **span** is a single unit of work or operation within a distributed system, representing a specific event or process with a start and end time. Spans are linked together to form traces, allowing engineers to follow requests as they propagate through different services.


---
## 📚 Part 3 — RAG Pipeline Tracing

A RAG pipeline has **3 stages**, each taking time and each capable of failing:

```
User Query
    │
    ▼
[Embed Query]        ← Gemini text-embedding API call
    │
    ▼
[Retrieve Docs]      ← FAISS similarity search (in-memory)
    │
    ▼
[Generate Answer]    ← LLM API call (Groq llama-3.3-70b)
    │
    ▼
Response
```

Without observability: *"It's slow"* — but which stage? The embedding? The retrieval? The LLM?
With Logfire: you see **exactly** which stage takes how long and what data flows through each one.

**Stack:**
- Embeddings: `gemini-embedding-2-preview` via Gemini API — no local model, reuses your existing key
- Vector store: `FAISS` — in-memory, no server needed
- LLM: Groq `llama-3.3-70b-versatile`